# 06 · Split-conformal calibration (PRD §10.3)
**Runs:** locally. **Produces:** `data/models/conformal_qhat.json`.

Checklist enforced in code: calibration set product-disjoint from training; built with
API-like (fully imputed) metadata; coverage on the held-out TEST split ≥ nominal − 2%
before anything is written.

In [ ]:
# --- Repo setup: make `app` (backend) and `vera_training` importable ---------
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "PRD.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ("backend", "training"):
    if str(ROOT / sub) not in sys.path:
        sys.path.insert(0, str(ROOT / sub))

import os
if not (ROOT / ".env").exists():  # app settings require these; real values come from .env
    os.environ.setdefault("DATABASE_URL", "sqlite:///:memory:")
    os.environ.setdefault("REDIS_URL", "redis://localhost:6379/0")

from vera_training import paths
paths.ensure_dirs()
print("Repo root:", ROOT)

In [ ]:
import joblib
import pandas as pd
from app.ml.agents.fusion import FusionModel
from vera_training import calibration
from vera_training.splits import assert_disjoint

cache = joblib.load(paths.FUSION_CACHE)["matrices"]
assert_disjoint(cache["train"]["frame"].parent_asin, cache["calibration"]["frame"].parent_asin,
                cache["test"]["frame"].parent_asin)
fusion = FusionModel(paths.MODELS / "fusion_ensemble.json")

def predict(split):
    members = fusion.predict_members(cache[split]["X"])
    return members.mean(axis=0).clip(1, 5), members.std(axis=0), cache[split]["y"]

cal, test = predict("calibration"), predict("test")

In [ ]:
rows = []
for alpha in (0.05, 0.10, 0.20):
    payload = calibration.fit_best_conformal(*cal, alpha=alpha)
    metrics = calibration.interval_metrics(*test, payload)
    rows.append({"target": 1 - alpha, "method": payload["method"], "q_hat": payload["q_hat"],
                 "test_coverage": metrics["coverage"], "test_mean_width": metrics["mean_width"],
                 "ok": metrics["coverage"] >= 1 - alpha - 0.02})
    if alpha == 0.10:
        production, production_test = payload, metrics
pd.DataFrame(rows)

In [ ]:
calibration.save_conformal(production, paths.MODELS / "conformal_qhat.json",
                           test_metrics=production_test)
print("Saved", paths.MODELS / "conformal_qhat.json", production)

# Optional: record in the CALIBRATION_CHECKPOINTS table (uses DATABASE_URL from .env;
# run `alembic upgrade head` against that database first).
# print("checkpoint id:", calibration.record_checkpoint(production))

## Upload all artifacts to HuggingFace Hub
Requires `HF_TOKEN` and `HF_MODEL_REPO` in `.env`. Refuses to upload an incomplete set.

In [ ]:
from dotenv import load_dotenv
from vera_training.hub import upload_artifacts

load_dotenv(ROOT / ".env")
uploaded = upload_artifacts(paths.MODELS, os.environ["HF_MODEL_REPO"], os.environ["HF_TOKEN"])
print("uploaded:", uploaded)